
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lecture - Change Data Capture (CDC) Review

#### Duration: ~7-10 Minutes

This lecture provides a comprehensive review of Change Data Capture (CDC) concepts and implementation patterns in the Lakehouse. You'll explore how CDC enables real-time data synchronization and learn about different approaches to handling changing data.

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Define** Change Data Capture (CDC) and explain its role in data synchronization
2. **Distinguish** between SCD Type 1 and SCD Type 2 patterns and their use cases
3. **Analyze** how SCD Type 1 overwrites existing data while SCD Type 2 preserves history
4. **Identify** when to use each SCD type based on business requirements
5. **Recognize** how `AUTO CDC INTO` simplifies CDC implementation in Apache Spark™ Declarative Pipelines(SDP)

## A. What is Change Data Capture?

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 470" width="100%" role="img" aria-label="Change data capture: inserts, updates and deletes from a source table are applied to a target table as SCD Type 1 or Type 2" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1000px;display:block;margin:12px auto;"><defs><marker id="co-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="co-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="co-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="co-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="co-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker><filter id="co-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><text x="14" y="36" font-size="21" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>•  Change Data Capture (CDC) is a technique </tspan><tspan font-weight="700">used to track and capture</tspan></text><text x="46" y="66" font-size="21" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">changes in a data source</tspan><tspan> (such as a database, lakehouse or data warehouse).</tspan></text><text x="14" y="108" font-size="21" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>•  Then </tspan><tspan font-weight="700">applying those changes</tspan><tspan> into the Lakehouse or data warehouse</tspan></text><text x="150" y="168" font-size="20" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Source Data</text><rect x="40" y="180" width="250" height="262" rx="0" fill="#fff" stroke="#1B5062" stroke-width="1.4" /><rect x="62" y="198" width="206" height="44" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="165" y="228" font-size="20" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >INSERTS</text><rect x="62" y="254" width="206" height="44" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="165" y="284" font-size="20" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >UPDATES</text><rect x="62" y="310" width="206" height="44" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="165" y="340" font-size="20" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >DELETES</text><text x="165" y="392" font-size="12.5" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >A Source table of changes, currently</text><text x="165" y="410" font-size="12.5" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >this must be streaming table</text><text x="312" y="198" font-size="15.5" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>CDC is categorized into different types of </tspan><tspan font-weight="700">Slowly Changing Dimensions</tspan></text><text x="312" y="220" font-size="15.5" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">(SCD)</tspan><tspan>, which represent how </tspan><tspan font-weight="700">historical changes are handled</tspan></text><path d="M300,290 H706" fill="none" stroke="#FF3621" stroke-width="2.4" marker-end="url(#co-ah-l)" stroke-linecap="round" stroke-linejoin="round"/><text x="520" y="336" font-size="20" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Main types</text><rect x="352" y="352" width="140" height="44" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="422" y="381" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >SCD Type 1</text><rect x="508" y="352" width="140" height="44" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="578" y="381" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >SCD Type 2</text><path d="M742,262 v92 a52,16 0 0 0 104,0 v-92" fill="#fff" stroke="#1B5062" stroke-width="3"/><ellipse cx="794" cy="262" rx="52" ry="16" fill="#fff" stroke="#1B5062" stroke-width="3"/><path d="M742,290 a52,16 0 0 0 104,0 M742,320 a52,16 0 0 0 104,0" fill="none" stroke="#1B5062" stroke-width="3"/><rect x="725" y="340" width="31" height="31" rx="3" fill="#fff" stroke="none" stroke-width="0" /><svg x="718" y="339" width="40" height="36.4" viewBox="0 0 56 51" fill="#1B5062"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B5062" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#1B5062" stroke-width="2.6" stroke-linecap="round"/></svg><text x="800" y="408" font-size="24" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Target Table</text><text x="800" y="430" font-size="12.5" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Table for changes to be</text><text x="800" y="446" font-size="12.5" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >applied</text></svg>



##### EXPAND FOR ADDITIONAL NOTES

<details>

Let's review **Change Data Capture (CDC)**, a foundational concept for keeping data synchronized across systems.

#### CDC Definition and Purpose

Change Data Capture is a technique used to track and capture changes in data sources like databases, Lakehouses, or data warehouses, and then apply those changes to a target table to ensure it reflects the latest state of the source.

#### Slowly Changing Dimensions (SCDs)

CDC is closely tied to the concept of **Slowly Changing Dimensions (SCDs)**, which describe how historical data changes are handled in your target system.

We'll focus on two main types:
- **SCD Type 1** - Overwrites existing data with new values (no history tracking)  
- **SCD Type 2** - Preserves history by storing previous versions of records

#### Real-World Example

Imagine a **customer** table where new customers are added, existing customer information is updated, or some customers are deleted. CDC ensures those changes flow into the target table using either SCD Type 1 or Type 2 logic, keeping the target table continuously up to date.

**Think about it:** What types of data changes occur frequently in your organization that would benefit from automated CDC processing?

#### Documentation Resources
- [What is change data capture (CDC)?](https://docs.databricks.com/aws/en/ldp/what-is-change-data-capture)

</details>


## B. Slowly Changing Dimensions (SCD) Type 1

### B1. SCD Type 1 - Overview

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 460" width="100%" role="img" aria-label="SCD Type 1 overwrites the target row with the latest value by key and removes deleted keys" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1000px;display:block;margin:12px auto;"><defs><marker id="s1-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="s1-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="s1-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="s1-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="s1-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker><filter id="s1-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><rect x="10" y="22" width="450" height="54" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="22" y="58" font-size="17" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >SCD Type 1 (Overwrite Target with Latest Values)</text><circle cx="22" cy="112" r="4" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="44" y="118" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>When a record updates, the </tspan><tspan font-weight="700">previous record is</tspan></text><text x="44" y="146" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">simply overwritten</tspan><tspan> by its </tspan><tspan font-weight="700">key</tspan><tspan> with the new value.</tspan></text><circle cx="22" cy="202" r="4" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="44" y="208" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>When a record is deleted by its </tspan><tspan font-weight="700">key</tspan><tspan>, the </tspan><tspan font-weight="700">record is</tspan></text><text x="44" y="236" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">removed</tspan></text><circle cx="22" cy="292" r="4" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="44" y="298" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>There is </tspan><tspan font-weight="700">no tracking of old keys (rows)</tspan><tspan>, only the</tspan></text><text x="44" y="326" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">current data</tspan><tspan> is retained.</tspan></text><text x="480" y="40" font-size="17" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>customers (</tspan><tspan font-style="italic">target table with all customers</tspan><tspan>)</tspan></text><rect x="480" y="52" width="120" height="34" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="489" y="74.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="600" y="52" width="120" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="609" y="74.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="720" y="52" width="135" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="729" y="74.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="855" y="52" width="145" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="864" y="74.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="480" y="86" width="520" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="489" y="107.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="609" y="107.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="729" y="107.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1 Blue Rd.</text><text x="864" y="107.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><rect x="480" y="118" width="520" height="32" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="489" y="139.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="609" y="139.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Samarth</text><text x="729" y="139.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">22 Front St</text><text x="864" y="139.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><path d="M520,168 L538.0,198.24 H526.48 V240 H513.52 V198.24 H502.0 z" fill="#1B5062"/><rect x="560" y="168" width="410" height="62" fill="#FFC95C" stroke="#0B2026" stroke-width="1.6" stroke-dasharray="2 4"/><text x="765.0" y="194.175" font-size="15.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-style="italic">Update the </tspan><tspan font-weight="700" font-style="italic">customers</tspan><tspan font-style="italic"> table with the </tspan><tspan font-weight="700" font-style="italic">new</tspan></text><text x="765.0" y="214.675" font-size="15.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700" font-style="italic">customer</tspan><tspan font-style="italic"> information</tspan></text><text x="480" y="268" font-size="17" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>updates (</tspan><tspan font-style="italic">source table with updates</tspan><tspan>)</tspan></text><rect x="480" y="280" width="100" height="34" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="489" y="302.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="580" y="280" width="90" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="589" y="302.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="670" y="280" width="125" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="679" y="302.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="795" y="280" width="95" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="804" y="302.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Operation</text><rect x="890" y="280" width="110" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="899" y="302.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="480" y="314" width="520" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="489" y="335.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="589" y="335.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="679" y="335.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>99 Sunny St..</tspan></text><text x="804" y="335.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='16'>UPDATE</tspan></text><text x="899" y="335.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/15/2025</text><rect x="480" y="346" width="520" height="32" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="489" y="367.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="589" y="367.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="679" y="367.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>123 Main St.</tspan></text><text x="804" y="367.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='16'>UPDATE</tspan></text><text x="899" y="367.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="480" y="378" width="520" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="489" y="399.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="589" y="399.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><text x="679" y="399.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><text x="804" y="399.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='16'>DELETE</tspan></text><text x="899" y="399.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="480" y="410" width="520" height="32" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="489" y="431.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="589" y="431.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Kostas</text><text x="679" y="431.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">100 Athens Dr.</text><text x="804" y="431.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='16'>NEW</tspan></text><text x="899" y="431.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text></svg>

##### EXPAND FOR ADDITIONAL NOTES

<details>


SCD Type 1 updates the **target table** by **overwriting existing rows** with the latest values. No version history is maintained - only the current state matters.

#### Key Characteristics:
- **Updates:** When a record is updated by its key(s), the existing row is replaced with new values
- **Deletes:** When a record is deleted by its key(s), it is removed from the target table  
- **Current State Only:** Only the most recent version of each record is stored
- **No History:** Previous changes and historical versions are not retained

#### Our Scenario Setup

**Target Table (customers)** - Current customer data:

| CustomerID | Name   | Address | ProcessDate |
|------------|--------|---------|-------------|
| 1 | Peter | 1 Blue Rd. | 5/1/2025 |
| 2 | Samarth | 22 Front St | 5/1/2025 |

**Source Updates** - Incoming changes:

| Change Type | Details |
|-------------|---------|
| **Update** | Peter has two address updates (5/15 and 5/20) for `customer_id = 1` |
| **Delete** | Samarth requests account removal (`customer_id = 2`) |
| **Insert** | New customer Kostas joins (`customer_id = 3`) |

**Goal:** Apply updates so the target table reflects the latest state of all customers.

</details>


### B2. SCD Type 1 - Implementation Example

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1040 530" width="100%" role="img" aria-label="SCD Type 1 example: updates, a delete and an insert are applied to the customers table, leaving only the current rows" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1040px;display:block;margin:12px auto;"><defs><marker id="si-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="si-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="si-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="si-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="si-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker><filter id="si-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><text x="190" y="32" font-size="17" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>customers (</tspan><tspan font-style="italic">target table with all customers</tspan><tspan>)</tspan></text><rect x="190" y="44" width="104" height="34" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="199" y="66.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="294" y="44" width="92" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="303" y="66.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="386" y="44" width="118" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="395" y="66.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="504" y="44" width="106" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="513" y="66.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="190" y="78" width="420" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="199" y="99.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="303" y="99.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="395" y="99.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1 Blue Rd.</text><text x="513" y="99.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><rect x="190" y="110" width="420" height="32" rx="0" fill="#FBC4BF" stroke="#E4E7EA" stroke-width="0.8" /><text x="199" y="131.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="303" y="131.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Samarth</text><text x="395" y="131.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">22 Front St</text><text x="513" y="131.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><rect x="190" y="142" width="420" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="199" y="163.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="303" y="163.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" font-style="italic">Kostas</text><text x="395" y="163.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" font-style="italic">100 Athens Dr.</text><text x="513" y="163.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" font-style="italic">5/20/2025</text><rect x="10" y="68" width="150" height="30" rx="0" fill="#FFAB00" stroke="none" stroke-width="1.5" /><text x="85" y="89" font-size="15" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Update the row</text><path d="M160,83 H186" fill="none" stroke="#FFAB00" stroke-width="2" marker-end="url(#si-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="10" y="100" width="150" height="30" rx="0" fill="#8F1737" stroke="none" stroke-width="1.5" /><text x="85" y="121" font-size="15" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delete the row</text><path d="M160,115 H186" fill="none" stroke="#8F1737" stroke-width="2" marker-end="url(#si-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="10" y="132" width="150" height="30" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="85" y="153" font-size="15" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Insert the row</text><path d="M160,147 H186" fill="none" stroke="#00A972" stroke-width="2" marker-end="url(#si-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><path d="M240,172 L256.0,199.72 H245.76 V238 H234.24 V199.72 H224.0 z" fill="#1B5062"/><text x="100" y="276" font-size="17" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>updates (</tspan><tspan font-style="italic">source table with updates</tspan><tspan>)</tspan></text><rect x="100" y="288" width="104" height="34" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="109" y="310.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="204" y="288" width="84" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="213" y="310.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="288" y="288" width="122" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="297" y="310.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="410" y="288" width="96" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="419" y="310.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Operation</text><rect x="506" y="288" width="104" height="34" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="515" y="310.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="100" y="322" width="510" height="32" rx="0" fill="#FFF1CC" stroke="#E4E7EA" stroke-width="0.8" /><text x="109" y="343.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="213" y="343.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="297" y="343.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>99 Sunny St..</tspan></text><text x="419" y="343.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='15.5'>UPDATE</tspan></text><text x="515" y="343.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/15/2025</text><rect x="100" y="354" width="510" height="32" rx="0" fill="#FFF1CC" stroke="#E4E7EA" stroke-width="0.8" /><text x="109" y="375.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="213" y="375.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="297" y="375.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>123 Main St.</tspan></text><text x="419" y="375.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='15.5'>UPDATE</tspan></text><text x="515" y="375.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="100" y="386" width="510" height="32" rx="0" fill="#FBC4BF" stroke="#E4E7EA" stroke-width="0.8" /><text x="109" y="407.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="213" y="407.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><text x="297" y="407.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><text x="419" y="407.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='15.5'>DELETE</tspan></text><text x="515" y="407.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="100" y="418" width="510" height="32" rx="0" fill="#00A972" stroke="#E4E7EA" stroke-width="0.8" /><text x="109" y="439.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="213" y="439.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Kostas</text><text x="297" y="439.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">100 Athens Dr.</text><text x="419" y="439.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='15.5'>NEW</tspan></text><text x="515" y="439.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><path d="M614,66 H628 V250 M614,372 H628 V250 H683" fill="none" stroke="#00A972" stroke-width="2" marker-end="url(#si-ah-k)" stroke-linecap="round" stroke-linejoin="round"/><rect x="686" y="130" width="348" height="270" rx="0" fill="#fff" stroke="#00A972" stroke-width="2.4" /><text x="700" y="166" font-size="19" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Updated SCD Type 1 Table</text><rect x="694" y="182" width="84" height="32" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="703" y="203.12" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="778" y="182" width="56" height="32" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="787" y="203.12" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="834" y="182" width="98" height="32" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="843" y="203.12" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="932" y="182" width="92" height="32" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="941" y="203.12" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="694" y="214" width="330" height="30" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="703" y="233.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="787" y="233.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="843" y="233.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>123 Main St.</tspan></text><text x="941" y="233.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="694" y="244" width="330" height="30" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="703" y="263.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="787" y="263.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Kostas</text><text x="843" y="263.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">100 Athens Dr.</text><text x="941" y="263.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><circle cx="706" cy="296" r="3.5" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="720" y="301" font-size="14" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">Peter</tspan><tspan> was updated with the </tspan><tspan font-weight="700">latest</tspan><tspan> information</tspan></text><circle cx="706" cy="324" r="3.5" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="720" y="329" font-size="14" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">Samarth</tspan><tspan> was deleted from the table</tspan></text><circle cx="706" cy="352" r="3.5" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="720" y="357" font-size="14" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">Kostas</tspan><tspan> was added to the table</tspan></text><rect x="80" y="470" width="840" height="44" rx="0" fill="#E8EAF6" stroke="#6C7A98" stroke-width="1" /><text x="500" y="499" font-size="17.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Used </tspan><tspan font-weight="700">SCD Type 1</tspan><tspan> when only the </tspan><tspan font-weight="700">current data matters</tspan><tspan> and historical values are irrelevant</tspan></text></svg>

> **NOTE:** When ingesting a `DELETE` row using `AUTO CDC INTO`, non key columns may contain `NULL` values. The required key columns must always be populated, as they are used to identify the row to delete.




##### EXPAND FOR ADDITIONAL NOTES

<details>

When we apply **SCD Type 1**, the target table is updated with the latest customer information using the **CustomerID** (key column) and **ProcessDate** (sequence column) to determine the most recent changes.

#### Processing Logic:

1. **Peter (CustomerID 1):** 
   - Multiple address updates exist (5/15 and 5/20)
   - Only the latest update (5/20) with address *123 Main St.* is applied
   - Previous address history is lost

2. **Samarth (CustomerID 2):** 
   - Marked for deletion
   - Entire row is removed from the target table
   - No trace of the customer remains

3. **Kostas (CustomerID 3):** 
   - New customer record
   - Inserted as a new row in the target table

#### Final Result:
The customers table contains only the most current snapshot of active customers. This approach is ideal when:
- Historical data is not required for business operations
- Storage efficiency is prioritized
- Regulatory compliance doesn't require audit trails

**Use SCD Type 1 when:** You only need the most up-to-date and accurate information without historical context.

#### Documentation
- [Use SCD Type 1 to keep only the latest data](https://docs.databricks.com/aws/en/ldp/what-is-change-data-capture#step-2-use-scd-type-1-to-keep-only-the-latest-data)

</details>


## C. Slowly Changing Dimensions (SCD) Type 2

### C1. SCD Type 2 - Scenario

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 460" width="100%" role="img" aria-label="SCD Type 2 scenario: an update, a delete and a new customer to apply to the customers table" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1000px;display:block;margin:12px auto;"><defs><marker id="s2s-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="s2s-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="s2s-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="s2s-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="s2s-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker><filter id="s2s-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><text x="110" y="40" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>customers (</tspan><tspan font-style="italic">target table with all customers</tspan><tspan>)</tspan></text><rect x="104" y="54" width="96" height="34" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="113" y="76.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="200" y="54" width="100" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="209" y="76.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="300" y="54" width="124" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="309" y="76.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="424" y="54" width="126" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="433" y="76.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="104" y="88" width="446" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="113" y="109.12" font-size="14.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="209" y="109.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="309" y="109.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>1 Blue Rd.</tspan></text><text x="433" y="109.12" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><rect x="104" y="120" width="446" height="32" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="113" y="141.12" font-size="14.5" font-weight="700" fill="#1B5062" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="209" y="141.12" font-size="14.5" font-weight="400" fill="#1B5062" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Samarth</text><text x="309" y="141.12" font-size="14.5" font-weight="400" fill="#1B5062" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">22 Front St</text><text x="433" y="141.12" font-size="14.5" font-weight="400" fill="#1B5062" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><path d="M122,160 L142.0,191.92 H129.2 V236 H114.8 V191.92 H102.0 z" fill="#1B5062"/><rect x="150" y="170" width="344" height="66" fill="#FFC95C" stroke="#0B2026" stroke-width="1.6" stroke-dasharray="2 4"/><text x="322.0" y="198.1" font-size="16" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-style="italic">Update the </tspan><tspan font-weight="700" font-style="italic">customers</tspan><tspan font-style="italic"> table with the </tspan><tspan font-weight="700" font-style="italic">new</tspan></text><text x="322.0" y="219.1" font-size="16" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700" font-style="italic">customer</tspan><tspan font-style="italic"> information for </tspan><tspan font-weight="700" font-style="italic">SCD Type 2</tspan></text><text x="104" y="292" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>updates (</tspan><tspan font-style="italic">source table with updates</tspan><tspan>)</tspan></text><rect x="102" y="306" width="96" height="36" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="111" y="329.76" font-size="15" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="198" y="306" width="100" height="36" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="207" y="329.76" font-size="15" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="298" y="306" width="124" height="36" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="307" y="329.76" font-size="15" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="422" y="306" width="100" height="36" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="431" y="329.76" font-size="15" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Operation</text><rect x="522" y="306" width="110" height="36" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="531" y="329.76" font-size="15" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="102" y="342" width="530" height="34" rx="0" fill="#FFF1CC" stroke="#E4E7EA" stroke-width="0.8" /><text x="111" y="364.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="207" y="364.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="307" y="364.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>123 Main St.</tspan></text><text x="431" y="364.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='17'>UPDATE</tspan></text><text x="531" y="364.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>5/20/2025</tspan></text><rect x="102" y="376" width="530" height="34" rx="0" fill="#FBC4BF" stroke="#E4E7EA" stroke-width="0.8" /><text x="111" y="398.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="207" y="398.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><text x="307" y="398.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><text x="431" y="398.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='17'>DELETE</tspan></text><text x="531" y="398.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="102" y="410" width="530" height="34" rx="0" fill="#00A972" stroke="#E4E7EA" stroke-width="0.8" /><text x="111" y="432.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="207" y="432.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Kostas</text><text x="307" y="432.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">100 Athens Dr.</text><text x="431" y="432.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700' font-size='17'>NEW</tspan></text><text x="531" y="432.44" font-size="15" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text></svg>

### C2. SCD Type 2 - Implementation
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1020 566" width="100%" role="img" aria-label="SCD Type 2 example table: current rows have a null end date, historical rows have an end date" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1020px;display:block;margin:12px auto;"><defs><marker id="s2i-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="s2i-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="s2i-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="s2i-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="s2i-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker><filter id="s2i-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><rect x="36" y="14" width="400" height="40" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="48" y="42" font-size="17" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >SCD Type 2 (Historical Tracking/Versioning)</text><circle cx="76" cy="86" r="4" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="100" y="92" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-style="italic">When a record changes:</tspan></text><circle cx="112" cy="120" r="3" fill="none" stroke="none" stroke-width="1.5"/><circle cx="112" cy="118" r="3.5" fill="none" stroke="#0B2026" stroke-width="1.6"/><text x="130" y="124" font-size="15.5" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>The </tspan><tspan font-weight="700">old record is preserved</tspan><tspan> with an additional column indicating its validity period (start date, end</tspan></text><text x="130" y="146" font-size="15.5" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>date, or a current flag) and a </tspan><tspan font-weight="700">new record is inserted</tspan><tspan> with the updated information.</tspan></text><circle cx="112" cy="176" r="3.5" fill="none" stroke="#0B2026" stroke-width="1.6"/><text x="130" y="182" font-size="15.5" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>When a </tspan><tspan font-weight="700">record is marked as deleted,</tspan><tspan> the record is kept and a column indicates the record is </tspan><tspan font-weight="700">inactive</tspan><tspan>.</tspan></text><circle cx="76" cy="226" r="4" fill="#0B2026" stroke="none" stroke-width="1.5"/><text x="100" y="232" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Used when </tspan><tspan font-weight="700">historical data is important</tspan><tspan>, and the system needs to track </tspan><tspan font-weight="700">how attributes change</tspan></text><text x="100" y="258" font-size="18" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">over time</tspan><tspan> (like tracking changes in a customer's address or status).</tspan></text><text x="40" y="336" font-size="19" font-weight="700" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Example customers target table with SCD Type 2</text><rect x="500" y="296" width="200" height="60" fill="#00A972" stroke="none"/><text x="600.0" y="321.325" font-size="14.5" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Columns indicate </tspan><tspan font-weight="700">active</tspan></text><text x="600.0" y="340.825" font-size="14.5" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>and </tspan><tspan font-weight="700">inactive</tspan><tspan> rows</tspan></text><path d="M562,358 V372" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#s2i-ah-k)" stroke-linecap="round" stroke-linejoin="round"/><path d="M648,358 V372" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#s2i-ah-k)" stroke-linecap="round" stroke-linejoin="round"/><rect x="36" y="374" width="104" height="34" rx="0" fill="#1273C5" stroke="none" stroke-width="1.5" /><text x="45" y="396.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="140" y="374" width="100" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="149" y="396.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="240" y="374" width="122" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="249" y="396.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="362" y="374" width="118" height="34" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="371" y="396.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="480" y="374" width="118" height="34" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="489" y="396.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >__START_AT</text><rect x="598" y="374" width="118" height="34" rx="0" fill="#00A972" stroke="none" stroke-width="1.5" /><text x="607" y="396.44" font-size="14.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >__END_AT</text><rect x="36" y="408" width="680" height="34" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="45" y="430.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="149" y="430.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="249" y="430.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">123 Main St.</text><text x="371" y="430.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><text x="489" y="430.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><text x="607" y="430.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><rect x="36" y="442" width="680" height="34" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="45" y="464.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="149" y="464.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="249" y="464.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1 Blue Rd.</text><text x="371" y="464.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><text x="489" y="464.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><text x="607" y="464.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="36" y="476" width="680" height="34" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="45" y="498.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="149" y="498.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Samarth</text><text x="249" y="498.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">22 Front St</text><text x="371" y="498.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><text x="489" y="498.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><text x="607" y="498.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="36" y="510" width="680" height="34" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="45" y="532.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="149" y="532.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Kostas</text><text x="249" y="532.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">100 Athens Dr.</text><text x="371" y="532.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><text x="489" y="532.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><text x="607" y="532.44" font-size="14.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">null</text><rect x="38" y="411" width="670" height="28" fill="none" stroke="#00A972" stroke-width="2.4"/><rect x="38" y="445" width="670" height="62" fill="none" stroke="#FF5F46" stroke-width="2.4"/><rect x="38" y="513" width="670" height="28" fill="none" stroke="#00A972" stroke-width="2.4"/><rect x="734" y="396" width="252" height="56" fill="#00A972" stroke="none"/><text x="860.0" y="419.475" font-size="13.5" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">Null</tspan><tspan> values indicate the </tspan><tspan font-weight="700">current</tspan></text><text x="860.0" y="437.975" font-size="13.5" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">row</tspan></text><rect x="734" y="462" width="252" height="62" fill="#FF5F46" stroke="none"/><text x="860.0" y="488.475" font-size="13.5" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">Non null</tspan><tspan> values indicate </tspan><tspan font-weight="700">inactive</tspan></text><text x="860.0" y="506.975" font-size="13.5" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">(historic)</tspan><tspan> rows</tspan></text><path d="M734,424 H714" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#s2i-ah-k)" stroke-linecap="round" stroke-linejoin="round"/><path d="M734,492 H714" fill="none" stroke="#FF5F46" stroke-width="1.8" marker-end="url(#s2i-ah-l)" stroke-linecap="round" stroke-linejoin="round"/></svg>



##### EXPAND FOR ADDITIONAL NOTES

<details>

**Slowly Changing Dimensions Type 2 (SCD Type 2)** introduces **historical tracking and versioning** of records, preserving a complete audit trail of all changes over time.

#### Core Principles:

When a record changes:
- **Historical Preservation:** The old record is preserved with metadata columns showing its validity period
- **New Version Creation:** A new record is inserted with the updated information
- **Soft Deletes:** Deleted records remain in the table but are flagged as inactive

#### Metadata Columns Added:
- **__START_AT** - Timestamp when the row became active  
- **__END_AT** - Timestamp when the row became inactive  
  - `NULL` **value** = currently active record
  - **Non** `NULL` **value** = inactive/historical record

#### Detailed Example Breakdown:

**Customer ID 1 - Peter:**  
- **Two records exist** for Peter in the final table
- **Active record:** Shows Peter's current address with `__END_AT = NULL`
- **Historical record:** Preserves his previous address with `__END_AT` populated

**Customer ID 2 - Samarth:**  
- **Account deletion** processed as a soft delete
- **Original record remains** but `__END_AT` is populated with deletion timestamp
- **No new record created** since this was a deletion operation

**Customer ID 3 - Kostas:**  
- **New customer insertion** creates active record
- **__START_AT** marks when he joined, **__END_AT** remains NULL

#### Business Value:
SCD Type 2 enables complete historical analysis, allowing you to:
- Track how customer attributes evolved over time
- Perform point-in-time analysis for any historical date
- Maintain compliance with audit requirements
- Support advanced analytics on changing dimensions

**Use SCD Type 2 when:** Historical data tracking is essential for business intelligence, compliance, or analytical requirements.

#### Documentation
- [Use SCD Type 2 to keep historical data](https://docs.databricks.com/aws/en/ldp/what-is-change-data-capture#step-3-use-scd-type-2-to-keep-historical-data)

</details>


## D. Implementing CDC with `AUTO CDC INTO` in Spark Declarative Pipelines (SCD Type 1 Example)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1100 560" width="100%" role="img" aria-label="AUTO CDC INTO example with the target and source tables and an annotated SQL statement" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1100px;display:block;margin:12px auto;"><defs><marker id="ac-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="ac-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="ac-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="ac-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="ac-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker><filter id="ac-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter></defs><text x="10" y="34" font-size="15.5" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>customers (</tspan><tspan font-style="italic">target table with all customers</tspan><tspan>)</tspan></text><rect x="10" y="44" width="92" height="30" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="19" y="63.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="102" y="44" width="66" height="30" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="111" y="63.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="168" y="44" width="92" height="30" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="177" y="63.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="260" y="44" width="98" height="30" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="269" y="63.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="10" y="74" width="348" height="30" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="19" y="93.8" font-size="11.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="111" y="93.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="177" y="93.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1 Blue Rd.</text><text x="269" y="93.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><rect x="10" y="104" width="348" height="30" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="19" y="123.8" font-size="11.5" font-weight="700" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="111" y="123.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Samarth</text><text x="177" y="123.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">22 Front St</text><text x="269" y="123.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/1/2025</text><path d="M120,168 L137.0,195.72 H126.12 V234 H113.88 V195.72 H103.0 z" fill="#1B5062"/><text x="10" y="280" font-size="15.5" font-weight="400" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >updates (source table with updates)</text><rect x="10" y="290" width="92" height="30" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="19" y="309.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >CustomerID</text><rect x="102" y="290" width="64" height="30" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="111" y="309.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Name</text><rect x="166" y="290" width="92" height="30" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="175" y="309.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Address</text><rect x="258" y="290" width="80" height="30" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="267" y="309.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Operation</text><rect x="338" y="290" width="94" height="30" rx="0" fill="#FF6B52" stroke="none" stroke-width="1.5" /><text x="347" y="309.8" font-size="11.5" font-weight="700" fill="#fff" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >ProcessDate</text><rect x="10" y="320" width="422" height="30" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="19" y="339.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="111" y="339.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="175" y="339.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">99 Sunny St..</text><text x="267" y="339.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>UPDATE</tspan></text><text x="347" y="339.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/15/2025</text><rect x="10" y="350" width="422" height="30" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="19" y="369.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">1</text><text x="111" y="369.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Peter</text><text x="175" y="369.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">123 Main St.</text><text x="267" y="369.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>UPDATE</tspan></text><text x="347" y="369.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="10" y="380" width="422" height="30" rx="0" fill="#fff" stroke="#E4E7EA" stroke-width="0.8" /><text x="19" y="399.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">2</text><text x="111" y="399.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Samarth</text><text x="175" y="399.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">22 Front St</text><text x="267" y="399.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>DELETE</tspan></text><text x="347" y="399.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="10" y="410" width="422" height="30" rx="0" fill="#F4F5F6" stroke="#E4E7EA" stroke-width="0.8" /><text x="19" y="429.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">3</text><text x="111" y="429.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">Kostas</text><text x="175" y="429.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">100 Athens Dr.</text><text x="267" y="429.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif"><tspan font-weight='700'>NEW</tspan></text><text x="347" y="429.8" font-size="11.5" font-weight="400" fill="#0B2026" style="font-family:'DM Sans',Arial,Helvetica,sans-serif">5/20/2025</text><rect x="8" y="320" width="428" height="62" fill="none" stroke="#1B5062" stroke-width="2.4"/><rect x="456" y="20" width="410" height="336" rx="6" fill="#fff" stroke="#C9D0D5" stroke-width="1.2" /><text x="476" y="50" font-size="12.6" textLength="325.6" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#1A73E8">CREATE</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">OR</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">REFRESH</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">STREAMING</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">TABLE</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#B8860B">customers</tspan><tspan fill="#0B2026">;</tspan></text><text x="476" y="112" font-size="12.6" textLength="222.0" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#1A73E8">CREATE</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">FLOW</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#0B2026">scd_type_1_flow</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">AS</tspan></text><text x="476" y="143" font-size="12.6" textLength="170.2" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#1A73E8">AUTO</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">CDC</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">INTO</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#B8860B">customers</tspan></text><text x="476" y="174" font-size="12.6" textLength="155.4" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#0B2026">  </tspan><tspan fill="#1A73E8">FROM</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">STREAM</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#B8860B">updates</tspan></text><text x="476" y="205" font-size="12.6" textLength="140.6" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#0B2026">  </tspan><tspan fill="#1A73E8">KEYS</tspan><tspan fill="#0B2026"> (</tspan><tspan fill="#0B2026">CustomerID</tspan><tspan fill="#0B2026">)</tspan></text><text x="476" y="236" font-size="12.6" textLength="318.2" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#0B2026">  </tspan><tspan fill="#1A73E8">APPLY</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">AS</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">DELETE</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">WHEN</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#0B2026">operation</tspan><tspan fill="#0B2026"> = </tspan><tspan fill="#C92A18">"DELETE"</tspan></text><rect x="472" y="220" width="326.2" height="22" fill="none" stroke="#1B5062" stroke-width="1.4"/><text x="476" y="267" font-size="12.6" textLength="185.0" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#0B2026">  </tspan><tspan fill="#1A73E8">SEQUENCE</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">BY</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#0B2026">ProcessDate</tspan></text><text x="476" y="298" font-size="12.6" textLength="222.0" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#0B2026">  </tspan><tspan fill="#1A73E8">COLUMNS</tspan><tspan fill="#0B2026"> * </tspan><tspan fill="#1A73E8">EXCEPT</tspan><tspan fill="#0B2026"> (</tspan><tspan fill="#0B2026">operation</tspan><tspan fill="#0B2026">)</tspan></text><text x="476" y="329" font-size="12.6" textLength="170.2" lengthAdjust="spacing" style="font-family:Consolas,Monaco,monospace" xml:space="preserve"><tspan fill="#0B2026">  </tspan><tspan fill="#1A73E8">STORED</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">AS</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">SCD</tspan><tspan fill="#0B2026"> </tspan><tspan fill="#1A73E8">TYPE</tspan><tspan fill="#0B2026"> 1;</tspan></text><rect x="830" y="30.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="42.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Create </tspan><tspan>a </tspan><tspan>streaming </tspan><tspan>table, </tspan><tspan>then </tspan><tspan>use </tspan><tspan>AUTO </tspan><tspan>CDC </tspan></text><text x="961.0" y="55.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>to </tspan><tspan>populate </tspan><tspan>it </tspan></text><path d="M830,45 H811.6" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="123.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="135.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Apply </tspan><tspan>updates, </tspan><tspan>inserts </tspan><tspan>and </tspan><tspan>deletes </tspan><tspan>to </tspan><tspan>the </tspan></text><text x="961.0" y="148.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">target </tspan><tspan>table </tspan></text><path d="M830,138 H656.2" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="154.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="166.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Source </tspan><tspan>records </tspan><tspan>to </tspan><tspan>determine </tspan><tspan font-weight="700">updates, </tspan><tspan font-weight="700">deletes </tspan></text><text x="961.0" y="179.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">and </tspan><tspan font-weight="700">inserts </tspan></text><path d="M830,169 H641.4" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="185.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="197.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>The </tspan><tspan>column(s) </tspan><tspan>that </tspan><tspan font-weight="700">uniquely </tspan><tspan font-weight="700">identify </tspan><tspan font-weight="700">a </tspan><tspan font-weight="700">row </tspan></text><text x="961.0" y="210.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>in </tspan><tspan>the </tspan><tspan>source </tspan><tspan>and </tspan><tspan>target </tspan><tspan>tables </tspan></text><path d="M830,200 H626.6" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="216.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="228.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Specifies </tspan><tspan>when </tspan><tspan>a </tspan><tspan>CDC </tspan><tspan>event </tspan><tspan>should </tspan><tspan>be </tspan><tspan>treated </tspan></text><text x="961.0" y="241.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>as </tspan><tspan>a </tspan><tspan font-weight="700">DELETE </tspan><tspan>rather </tspan><tspan>than </tspan><tspan>an </tspan><tspan>upsert. </tspan></text><path d="M830,231 H804.2" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="247.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="259.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Specifies </tspan><tspan>the </tspan><tspan font-weight="700">logical </tspan><tspan font-weight="700">order </tspan><tspan font-weight="700">of </tspan><tspan font-weight="700">CDC </tspan><tspan font-weight="700">events </tspan><tspan>in </tspan></text><text x="961.0" y="272.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>the </tspan><tspan>source </tspan><tspan>data </tspan></text><path d="M830,262 H671.0" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="278.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="290.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Specifies </tspan><tspan>a </tspan><tspan font-weight="700">subset </tspan><tspan font-weight="700">of </tspan><tspan font-weight="700">columns </tspan><tspan>to </tspan><tspan>include </tspan><tspan>in </tspan></text><text x="961.0" y="303.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>the </tspan><tspan font-weight="700">target </tspan></text><path d="M830,293 H708.0" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="830" y="309.5" width="262" height="29" rx="0" fill="#1B5062" stroke="none" stroke-width="1.5" /><text x="961.0" y="321.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Whether </tspan><tspan>to </tspan><tspan>store </tspan><tspan>records </tspan><tspan>as </tspan><tspan font-weight="700">SCD </tspan><tspan font-weight="700">type </tspan><tspan font-weight="700">1 </tspan></text><text x="961.0" y="334.5" font-size="11" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-weight="700">(default) </tspan><tspan font-weight="700">or </tspan><tspan font-weight="700">type </tspan><tspan font-weight="700">2 </tspan></text><path d="M830,324 H656.2" fill="none" stroke="#1B5062" stroke-width="1.6" marker-end="url(#ac-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><text x="456" y="380" font-size="12.8" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-style="italic">INSERTs and UPDATEs are implicitly implemented using the </tspan><tspan font-weight="700" font-style="italic">KEY(s)</tspan><tspan font-style="italic">,</tspan></text><text x="456" y="397" font-size="12.8" fill="#0B2026" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan font-style="italic">no coding required</tspan></text><rect x="456" y="410" width="410" height="78" fill="#FFC95C" stroke="none"/><text x="661.0" y="435.225" font-size="13.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Use </tspan><tspan font-weight="700">AUTO CDC INTO (Formerly APPLY CHANGES INTO)</tspan></text><text x="661.0" y="453.725" font-size="13.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>instead of a traditional batch job with</tspan></text><text x="661.0" y="472.225" font-size="13.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>complex </tspan><tspan font-weight="700">MERGE INTO</tspan><tspan> syntax</tspan></text><rect x="456" y="496" width="410" height="62" fill="#FFC95C" stroke="none"/><text x="661.0" y="522.475" font-size="13.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>Use </tspan><tspan font-weight="700">AUTO CDC FROM SNAPSHOT</tspan><tspan> to process changes in</tspan></text><text x="661.0" y="540.975" font-size="13.5" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" xml:space="preserve" ><tspan>database snapshots.</tspan></text></svg>

> **NOTE:** The `AUTO CDC INTO` key can be defined using a single column or a composite key. A composite key simply means using multiple columns together to uniquely identify a record, for example (`CustomerID`, `OrderID`).


##### EXPAND FOR ADDITIONAL NOTES

<details>

Now that we've covered CDC concepts and both SCD patterns, let's explore how **Apache Spark™ Declarative Pipelines** simplifies CDC implementation with the `AUTO CDC INTO` statement.

**NOTE:** The `AUTO CDC` APIs were previously known as `APPLY CHANGES INTO`, but the syntax and functionality remain identical.

#### AUTO CDC INTO Syntax Breakdown

```sql
CREATE OR REFRESH STREAMING TABLE customers;

CREATE FLOW scd_type_1_flow AS
AUTO CDC INTO customers 
 FROM STREAM updates
 KEYS (CustomerID)                              
 APPLY AS DELETE WHEN operation = "DELETE"     
 SEQUENCE BY ProcessDate                 
 COLUMNS * EXCEPT (operation)  
 STORED AS SCD TYPE 1;
```

- **`AUTO CDC INTO customers`** - Specifies the target table for CDC operations
- **`FROM STREAM updates`** - Defines the source stream containing CDC events
- **`KEYS (CustomerID)`** - Establishes unique key(s) for matching source and target records
- **`APPLY AS DELETE WHEN operation = "DELETE"`** - Defines deletion logic based on operation column
- **`SEQUENCE BY ProcessDate`** - Ensures events are processed in chronological order
- **`COLUMNS * EXCEPT (operation)`** - Includes all columns except operational metadata
- **`STORED AS SCD TYPE 1`** - Specifies SCD Type 1 pattern (default is SCD Type 1 default)

#### Key Advantages

`AUTO CDC INTO` provides significant benefits over traditional approaches:

1. **Simplified Implementation:** Eliminates complex `MERGE INTO` logic
2. **Automatic Ordering:** Handles event sequencing automatically
3. **Built-in SCD Support:** Native support for both Type 1 and Type 2 patterns
4. **Streaming Integration:** Works seamlessly with both streaming and batch sources
5. **Error Handling:** Includes robust error handling and recovery mechanisms

**Reflection Question:** How might `AUTO CDC INTO` simplify your current data pipeline maintenance compared to custom merge logic?

</details>



## E. Documentation and Next Steps

#### Key Resources:
- [The AUTO CDC APIs: Simplify change data capture with Apache Spark™ Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/cdc)
- [AUTO CDC INTO (Apache Spark™ Declarative Pipelines)](https://docs.databricks.com/aws/en/ldp/developer/ldp-sql-ref-apply-changes-into)

### Coming Up Next: Automating SCD Type 2 with AUTO CDC in Apache Spark™ Declarative Pipelines

In the next notebook, you will apply what you just learned by building a pipeline that implements CDC with SCD Type 2 using `AUTO CDC` in Apache Spark™ Declarative Pipelines.

## F. Summary and Key Takeaways

### What We Covered:

1. **Change Data Capture (CDC)** enables automated synchronization of data changes between systems
2. **SCD Type 1** overwrites existing data, maintaining only current state (no history)
3. **SCD Type 2** preserves complete historical versions using metadata columns
4. **`AUTO CDC INTO`** provides declarative, simplified CDC implementation in Lakeflow pipelines

### Decision Framework:

**Choose SCD Type 1 when:**
- Only current data state is needed
- Storage efficiency is prioritized
- Historical tracking is not required

**Choose SCD Type 2 when:**
- Historical analysis is essential
- Audit trails are required for compliance
- Point-in-time reporting is needed

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>